In [2]:
# Copyright (c) 2024 Microsoft Corporation.
# Licensed under the MIT License.

In [3]:
import os

import pandas as pd
import tiktoken

from graphrag.query.indexer_adapters import read_indexer_entities, read_indexer_reports
from graphrag.query.llm.oai.chat_openai import ChatOpenAI
from graphrag.query.llm.oai.typing import OpenaiApiType
from graphrag.query.structured_search.global_search.community_context import (
    GlobalCommunityContext,
)
from graphrag.query.structured_search.global_search.search import GlobalSearch

/data/jiacheng/miniconda3/envs/common/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Global Search example

Global search method generates answers by searching over all AI-generated community reports in a map-reduce fashion. This is a resource-intensive method, but often gives good responses for questions that require an understanding of the dataset as a whole (e.g. What are the most significant values of the herbs mentioned in this notebook?).

### LLM setup

In [5]:
api_key = os.environ["OPENAI_API_KEY"]
llm_model = "gpt-4o-mini"

llm = ChatOpenAI(
    api_key=api_key,
    model=llm_model,
    api_type=OpenaiApiType.OpenAI,  # OpenaiApiType.OpenAI or OpenaiApiType.AzureOpenAI
    max_retries=20,
)

token_encoder = tiktoken.get_encoding("cl100k_base")

### Load community reports as context for global search

- Load all community reports in the `create_final_community_reports` table from the ire-indexing engine, to be used as context data for global search.
- Load entities from the `create_final_nodes` and `create_final_entities` tables from the ire-indexing engine, to be used for calculating community weights for context ranking. Note that this is optional (if no entities are provided, we will not calculate community weights and only use the `rank` attribute in the community reports table for context ranking)

In [10]:
# parquet files generated from indexing pipeline
INPUT_DIR = "/home/ljc/data/graphrag/alltest/ablation_new_1212/cyber_v3_tobeuse_only1_t3_shuffle/output/20241218-204530/artifacts"
COMMUNITY_REPORT_TABLE = "create_final_community_reports"
ENTITY_TABLE = "create_final_nodes"
ENTITY_EMBEDDING_TABLE = "create_final_entities"

# community level in the Leiden community hierarchy from which we will load the community reports
# higher value means we use reports from more fine-grained communities (at the cost of higher computation cost)
COMMUNITY_LEVEL = 2

In [11]:
entity_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_TABLE}.parquet")
report_df = pd.read_parquet(f"{INPUT_DIR}/{COMMUNITY_REPORT_TABLE}.parquet")
entity_embedding_df = pd.read_parquet(f"{INPUT_DIR}/{ENTITY_EMBEDDING_TABLE}.parquet")

reports = read_indexer_reports(report_df, entity_df, COMMUNITY_LEVEL)
entities = read_indexer_entities(entity_df, entity_embedding_df, COMMUNITY_LEVEL)
print(f"Total report count: {len(report_df)}")
print(
    f"Report count after filtering by community level {COMMUNITY_LEVEL}: {len(reports)}"
)
report_df.head()

Total report count: 207
Report count after filtering by community level 2: 158


,community,full_content,level,rank,title,rank_explanation,summary,findings,full_content_json,id
0,205,# Adversary-in-the-Middle Cybersecurity Commun...,3,8.5,Adversary-in-the-Middle Cybersecurity Community,The impact severity rating is high due to the ...,The community focuses on the Adversary-in-the-...,[{'explanation': 'Adversary-in-the-Middle is a...,"{\n ""title"": ""Adversary-in-the-Middle Cyber...",dae236cd-e6e4-4d5d-9c7f-c93c2c81b2c7
1,206,# DHCP Spoofing and Network Vulnerabilities\n\...,3,8.5,DHCP Spoofing and Network Vulnerabilities,The impact severity rating is high due to the ...,The community focuses on the cybersecurity thr...,[{'explanation': 'DHCP Spoofing is a significa...,"{\n ""title"": ""DHCP Spoofing and Network Vul...",6e140426-7e57-4e88-82aa-54dcfa916c5c
2,106,# Spear Phishing and Cyber Threat Actors\n\nTh...,2,8.5,Spear Phishing and Cyber Threat Actors,The impact severity rating is high due to the ...,The community focuses on the evolving techniqu...,[{'explanation': 'Spear phishing has emerged a...,"{\n ""title"": ""Spear Phishing and Cyber Thre...",64902258-69b2-41b2-8127-01ed96b2ebe0
3,107,"# Cyber Threat Actors: Carberp, Calisto, and T...",2,8.5,"Cyber Threat Actors: Carberp, Calisto, and Tor...",The impact severity rating is high due to the ...,This community comprises significant cyber thr...,[{'explanation': 'Carberp is a credential and ...,"{\n ""title"": ""Cyber Threat Actors: Carberp,...",245b921d-a276-4caf-8d24-2f7b0233989c
4,108,"# Cyber Threat Actors: Hildegard, Stonedrill, ...",2,8.5,"Cyber Threat Actors: Hildegard, Stonedrill, an...",The impact severity rating is high due to the ...,This community comprises sophisticated cyber t...,[{'explanation': 'Hildegard has significantly ...,"{\n ""title"": ""Cyber Threat Actors: Hildegar...",609acc26-f017-4cfc-921b-189ecdd1d1d0


#### Build global context based on community reports

In [12]:
context_builder = GlobalCommunityContext(
    community_reports=reports,
    entities=entities,  # default to None if you don't want to use community weights for ranking
    token_encoder=token_encoder,
)

#### Perform global search

In [13]:
context_builder_params = {
    "use_community_summary": False,  # False means using full community reports. True means using community short summaries.
    "shuffle_data": True,
    "include_community_rank": True,
    "min_community_rank": 0,
    "community_rank_name": "rank",
    "include_community_weight": True,
    "community_weight_name": "occurrence weight",
    "normalize_community_weight": True,
    "max_tokens": 12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
    "context_name": "Reports",
}

map_llm_params = {
    "max_tokens": 1000,
    "temperature": 0.0,
    "response_format": {"type": "json_object"},
}

reduce_llm_params = {
    "max_tokens": 2000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 1000-1500)
    "temperature": 0.0,
}

In [14]:
search_engine = GlobalSearch(
    llm=llm,
    context_builder=context_builder,
    token_encoder=token_encoder,
    max_data_tokens=12_000,  # change this based on the token limit you have on your model (if you are using a model with 8k limit, a good setting could be 5000)
    map_llm_params=map_llm_params,
    reduce_llm_params=reduce_llm_params,
    allow_general_knowledge=False,  # set this to True will add instruction to encourage the LLM to incorporate general knowledge in the response, which may increase hallucinations, but could be useful in some use cases.
    json_mode=True,  # set this to False if your LLM model does not support JSON mode.
    context_builder_params=context_builder_params,
    concurrent_coroutines=32,
    response_type="multiple paragraphs",  # free form text describing the response type and format, can be anything, e.g. prioritized list, single paragraph, multiple paragraphs, multiple-page report
)

In [15]:
result = await search_engine.asearch(
    "What is the major conflict in this story and who are the protagonist and antagonist?"
)

print(result.response)

## Major Conflict

The major conflict in this story revolves around the ongoing battle between cyber threat actors and cybersecurity professionals. This conflict is characterized by the evolving tactics and techniques employed by cybercriminals, such as malware entities like PUPY, CROSSRAT, and others, which continuously adapt to advancements in cybersecurity measures. The dynamic nature of these cyber threats necessitates that organizations remain vigilant and innovate their defensive strategies to mitigate risks effectively [Data: Reports (193, 115, 139, 45, 32, +more); Reports (33, 37, 114, 195, 117); Reports (108, 96, 145, 149, 60, +more)].

The conflict highlights the tension between malicious actors, who exploit vulnerabilities in systems, and the cybersecurity professionals tasked with protecting sensitive information. As cybercriminals refine their strategies, organizations must implement robust defenses to counteract sophisticated attacks, which may include advanced techniques

In [16]:
# inspect the data used to build the context for the LLM responses
result.context_data["reports"]

,id,title,occurrence weight,content,rank
0,193,PUPY and CROSSRAT Cyber Threat Community,1.000000,# PUPY and CROSSRAT Cyber Threat Community\n\n...,8.5
1,115,Multi-Factor Authentication and Cyber Threat M...,0.742424,# Multi-Factor Authentication and Cyber Threat...,7.5
2,139,"Cyber Threat Community: Emotet, Pony, and Lucifer",0.681818,"# Cyber Threat Community: Emotet, Pony, and Lu...",9.0
3,45,Cyber Threat Actors: P.A.S. Webshell and Xbash,0.545455,# Cyber Threat Actors: P.A.S. Webshell and Xba...,8.5
4,32,Cyber Threat Community: PowerSploit and Exaramel,0.545455,# Cyber Threat Community: PowerSploit and Exar...,8.5
...,...,...,...,...,...
153,130,Shared Modules and Cyber Threats,0.060606,# Shared Modules and Cyber Threats\n\nThe comm...,8.5
154,62,Elderwood Cybercriminal Community,0.060606,# Elderwood Cybercriminal Community\n\nThe Eld...,8.5
155,187,DarkWatchman and Cyber Threats in Mexico,0.045455,# DarkWatchman and Cyber Threats in Mexico\n\n...,8.5
156,183,Enhanced Logging and Real-Time Monitoring Solu...,0.045455,# Enhanced Logging and Real-Time Monitoring So...,7.5


In [17]:
# inspect number of LLM calls and tokens
print(f"LLM calls: {result.llm_calls}. LLM tokens: {result.prompt_tokens}")

LLM calls: 10. LLM tokens: 111487
